# Brute-Force Pairs Finder
This notebook implements the "brute-force" approach to finding tradable pairs. The process is:

Define a Universe: We'll select a group of related stocks (e.g., the S&P 500 Financials) to test.
Get Data: We'll use our DataHandler to fetch 3-5 years of daily price data for this universe.
Pair Up: We'll programmatically create every possible unique pair from this universe.
Test for Cointegration: For each pair, we will:
Run an OLS regression to find the hedge ratio (k).
Calculate the "spread" (the residuals from the regression).
Run the Augmented Dickey-Fuller (ADF) test on the spread to see if it's stationary.
Save Results: We'll save all pairs that pass the test (p-value < 0.05) to a list.

In [8]:
import pandas as pd
import numpy as np
import os
import itertools
from dotenv import load_dotenv

# --- Math & Stats Libraries ---
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller

# --- Alpaca & Data Handling ---
# We'll use the DataHandler class we built, assuming it's in the 'src' folder
# If running this from the 'notebooks' folder, we need to adjust the path
import sys
sys.path.append('../src') # This allows us to import from the 'src' folder
from data_handler import DataHandler

print("Libraries imported successfully.")

Libraries imported successfully.


In [9]:
# --- Step 1: Define Universe & Timeframe ---

# A targeted "brute-force" is better than testing thousands of stocks.
# Let's use a list of major US financial stocks (components of the XLF ETF).
# This provides a good fundamental reason for cointegration.
UNIVERSE = [
 'BLK', 'SPGI', 'AXP','JPM'
    'COF', 'USB','MCO','PYPL','V','MA'
]

# Cointegration is a long-term relationship, so we need several years of data.
START_DATE = '2020-01-01'
END_DATE = '2024-01-01' # Test on data up to the start of 2024
TIMEFRAME = '1D' # Daily data is standard for this

print(f"Testing {len(UNIVERSE)} stocks from {START_DATE} to {END_DATE}.")

Testing 9 stocks from 2020-01-01 to 2024-01-01.


In [10]:
# --- Step 2: Get & Prepare Price Data ---

print("Fetching historical data...")
dh = DataHandler(paper_trading=True)

# Get the dictionary of DataFrames from our handler
hist_data = dh.get_historical_bars(UNIVERSE, TIMEFRAME, START_DATE, END_DATE)

# We need to combine this into a single DataFrame of closing prices
print("Processing data into a master DataFrame...")
close_prices = pd.DataFrame()

for symbol in UNIVERSE:
    if hist_data and symbol in hist_data and not hist_data[symbol].empty:
        # We need to make sure the index is a DatetimeIndex
        df = hist_data[symbol]
        df.index = pd.to_datetime(df.index)
        
        # Get just the closing prices
        close_prices[symbol] = df['close']
    else:
        print(f"Warning: No data for {symbol}. It will be skipped.")

# Drop any rows with missing data for any stock
close_prices.dropna(inplace=True)

# Re-update our universe list to only include stocks we have data for
UNIVERSE = close_prices.columns.tolist()

print(f"Data prepared. Master DataFrame shape: {close_prices.shape}")
display(close_prices.head())

Fetching historical data...
Data Handler (alpaca-py) initialized.
Submitting data request to Alpaca...
Processing data into a master DataFrame...
Data prepared. Master DataFrame shape: (1006, 8)
Processing data into a master DataFrame...
Data prepared. Master DataFrame shape: (1006, 8)


,BLK,SPGI,AXP,USB,MCO,PYPL,V,MA
timestamp,,,,,,,,
2020-01-02 05:00:00+00:00,508.98,277.84,125.85,59.20,241.72,110.75,191.12,303.39
2020-01-03 05:00:00+00:00,503.57,276.91,124.60,58.51,241.12,108.76,189.60,300.43
2020-01-06 05:00:00+00:00,504.00,279.04,124.06,57.71,241.87,110.17,189.19,301.23
2020-01-07 05:00:00+00:00,507.22,280.98,123.41,57.16,241.00,109.67,188.69,300.21
2020-01-08 05:00:00+00:00,507.10,285.01,125.54,57.04,245.62,111.82,191.92,305.10


In [11]:
# --- Step 3: Cointegration Test Function ---

def find_cointegration(series_1, series_2):
    """
    Tests for cointegration between two price series.
    
    1. Runs OLS regression: series_1 = k * series_2 + intercept
    2. Calculates the spread (residuals).
    3. Runs ADF test on the spread.
    
    :return: (hedge_ratio, adf_p_value)
    """
    
    # 1. Run OLS regression
    # We add a constant (intercept) to the independent variable
    series_2_with_const = sm.add_constant(series_2)
    model = sm.OLS(series_1, series_2_with_const)
    results = model.fit()
    
    hedge_ratio = results.params[1] # 'k'
    
    # 2. Calculate the spread
    spread = series_1 - hedge_ratio * series_2
    
    # 3. Run ADF test on the spread
    # The null hypothesis of ADF is that the series IS non-stationary
    # We want a low p-value to reject the null hypothesis
    adf_test = adfuller(spread)
    adf_p_value = adf_test[1] # The p-value
    
    return hedge_ratio, adf_p_value

In [12]:
# --- Step 4: The Brute-Force Loop ---

print("Running cointegration tests on all pairs...")

# Set our significance threshold
P_VALUE_THRESHOLD = 0.05

# Create all unique pairs of stocks
all_pairs = list(itertools.combinations(UNIVERSE, 2))

cointegrated_pairs = []

for symbol_a, symbol_b in all_pairs:
    
    series_a = close_prices[symbol_a]
    series_b = close_prices[symbol_b]
    
    hedge_ratio, p_value = find_cointegration(series_a, series_b)
    
    if p_value < P_VALUE_THRESHOLD:
        print(f"Found Cointegrated Pair: {symbol_a} / {symbol_b} | p-value: {p_value:.4f} | hedge_ratio: {hedge_ratio:.2f}")
        cointegrated_pairs.append({
            'symbol_a': symbol_a,
            'symbol_b': symbol_b,
            'hedge_ratio': hedge_ratio,
            'p_value': p_value
        })

print("\n--- Test Complete ---")
print(f"Total pairs tested: {len(all_pairs)}")
print(f"Total cointegrated pairs found: {len(cointegrated_pairs)}")

Running cointegration tests on all pairs...
Found Cointegrated Pair: BLK / USB | p-value: 0.0274 | hedge_ratio: 9.07
Found Cointegrated Pair: BLK / USB | p-value: 0.0274 | hedge_ratio: 9.07


C:\Users\trash\AppData\Local\Temp\ipykernel_3472\1384126882.py:20: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  hedge_ratio = results.params[1] # 'k'
C:\Users\trash\AppData\Local\Temp\ipykernel_3472\1384126882.py:20: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  hedge_ratio = results.params[1] # 'k'
C:\Users\trash\AppData\Local\Temp\ipykernel_3472\1384126882.py:20: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  hedge_ratio = results

Found Cointegrated Pair: SPGI / MCO | p-value: 0.0232 | hedge_ratio: 1.10


C:\Users\trash\AppData\Local\Temp\ipykernel_3472\1384126882.py:20: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  hedge_ratio = results.params[1] # 'k'
C:\Users\trash\AppData\Local\Temp\ipykernel_3472\1384126882.py:20: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  hedge_ratio = results.params[1] # 'k'
C:\Users\trash\AppData\Local\Temp\ipykernel_3472\1384126882.py:20: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  hedge_ratio = results

Found Cointegrated Pair: MCO / V | p-value: 0.0393 | hedge_ratio: 1.66
Found Cointegrated Pair: V / MA | p-value: 0.0083 | hedge_ratio: 0.53

--- Test Complete ---
Total pairs tested: 28
Total cointegrated pairs found: 4


In [13]:
# --- Step 5: Analyze Results ---

# Convert the results into a clean DataFrame for analysis
results_df = pd.DataFrame(cointegrated_pairs)

print("\nCointegrated Pairs Summary:")

display(results_df)

# You can now save this to a file
# results_df.to_csv('cointegrated_pairs.csv', index=False)


Cointegrated Pairs Summary:


,symbol_a,symbol_b,hedge_ratio,p_value
0,BLK,USB,9.070770,0.027424
1,SPGI,MCO,1.102522,0.023248
2,MCO,V,1.659694,0.039320
3,V,MA,0.527159,0.008333


In [14]:
from pathlib import Path
OUTPUT_DIR = (Path.cwd().parent / 'data')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
output_path = OUTPUT_DIR / 'cointegrated_pairs.csv'
if 'results_df' not in globals():
    raise NameError("results_df is not defined. Run the cell that builds the summary DataFrame first.")
results_df.to_csv(output_path, index=False)
print(f"Saved {len(results_df)} cointegrated pairs to {output_path}")

Saved 4 cointegrated pairs to c:\Users\trash\trading-bot\data\cointegrated_pairs.csv
